# Agentic RAG
First of all, the agent decides whether the question is better suited for graph retrieval or vector is enough. Such examples are already in the ragas testset csv.

In [48]:
import pandas as pd
import os
import openai
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../../src").resolve()))
from config import API_GENERATION_MODEL,API_REFORMULATION_MODEL, USE_LOCAL_GENERATION, LOCAL_GENERATION_MODEL, USE_GRAPH
print(API_GENERATION_MODEL)
os.getcwd()

gpt-5-nano


'c:\\Users\\filip\\Desktop\\Thesis\\project\\notebooks\\pipelines'

In [44]:
def load_example(address, n_per_class=3):
    path = Path(address)
    df = pd.read_csv(path)
    questions = df[["user_input", "synthesizer_name"]]
    questions = questions.rename(columns={"user_input": "Question", "synthesizer_name": "Type"})
    questions = questions.replace("single_hop_specific_query_synthesizer", "simple")
    questions = questions.replace("multi_hop_abstract_query_synthesizer", "abstract")
    questions = questions.replace("multi_hop_specific_query_synthesizer", "relational")
    
    sampled = questions.groupby("Type").head(n_per_class).reset_index(drop=True)
    return sampled
load_example("../../outputs/ragas_testset.csv")

,Question,Type
0,Can you explain the significance of Scholz et ...,simple
1,What research did Siegelman H.W. contribute to...,simple
2,How is the AB28 primer utilized in the amplifi...,simple
3,How does the latticework organization of Marte...,abstract
4,What is the process of seaweed extract prepara...,abstract
5,What are the characteristics of the flabellate...,abstract
6,What is the taxonomic relationship between Gra...,relational
7,What toxic substances from microalgae were rep...,relational
8,What did Kim and Joo find about the zooplankto...,relational


In [ ]:
def classify_query(query, examples=load_example("../../outputs/ragas_testset.csv")):
    """
    Classify a query as SIMPLE, RELATIONAL, or ABSTRACT using few-shot examples
    drawn from the RAGAS evaluation testset.
    """
    from openai import OpenAI

    if USE_LOCAL_GENERATION:
        client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
        model = LOCAL_GENERATION_MODEL
    else:
        client = OpenAI(
            api_key=os.getenv("DEEPSEEK_API_KEY"),
            base_url="https://api.deepseek.com")
        model = API_REFORMULATION_MODEL
        

    example_lines = "\n".join(
        f'- "{row.Question}" → {row.Type.upper()}'
        for row in examples.itertuples()
    )

    response = client.chat.completions.create(
        model=model,
        messages=[{
            "role": "system",
            "content": f"""Classify the question as SIMPLE, RELATIONAL, or ABSTRACT.

SIMPLE: A question about a single entity with no comparisons, no relationships between things, and no 'how does X affect Y' structure is SIMPLE. If in doubt, classify as SIMPLE.
RELATIONAL: asks about a specific relationship, interaction, or comparison between named entities. Keywords like 'affect', 'relate to', 'compare', 'differ from', 'interact with' signal RELATIONAL.
ABSTRACT: asks about broad themes, trends, or summaries across an entire field or corpus. No specific entities named. Keywords like 'main themes', 'overview', 'general trends' signal ABSTRACT.

Examples from algae research:
{example_lines}

Respond with only one word: SIMPLE or RELATIONAL or ABSTRACT."""
        }, {
            "role": "user",
            "content": query
        }],
        max_completion_tokens=5,#one word
        temperature=0
    )

    result = response.choices[0].message.content.strip().upper()
    #print(f"Query: {query[:60]}...")
    #print(f"Classification: {result}")

    if "ABSTRACT" in result:
        return "ABSTRACT"
    elif "RELATIONAL" in result:
        return "RELATIONAL"
    else:
        return "SIMPLE"

In [57]:
#TEST
c1 = classify_query("What is Chlorella?")  # should be SIMPLE
c2 = classify_query("How does Ulva pertusa affect nitrogen uptake in eelgrass beds?")  # should be RELATIONAL
c3 = classify_query("What are the main research themes in Korean macroalgae studies?")  # should be ABSTRACT
print((c1,c2,c3))

('SIMPLE', 'ABSTRACT', 'ABSTRACT')


In [47]:
def route_query(query):
    """
    Decide whether to use graph expansion based on query complexity.
    Returns True (use graph) or False (vector only).
    If USE_ROUTER is False in config, falls back to the static USE_GRAPH toggle.
    """
    from config import USE_ROUTER, USE_GRAPH

    if not USE_ROUTER:
        return USE_GRAPH

    classification = classify_query(query)

    if classification == "RELATIONAL" or classification == "ABSTRACT":
        return True
    else:
        return False

In [39]:
#TEST
#route_query("What is Zostera Marina and what are its industrial applications in biofuel production?")
#route_query("How does the presence of Ulva pertusa in eelgrass beds affect nitrogen cycling in Korean coastal waters?")
route_query("What methods have been used to study the relationship between salinity and growth rates across different Gracilaria species?")

False